In [ ]:
from torch.nn import *
import torch
import numpy as np
from sklearn.datasets import make_classification

torch.set_default_tensor_type(torch.cuda.FloatTensor)


def train(x, y, log=0):
    m = Sequential(Linear(4, 4), Tanh(), Linear(4, 2))
    o = torch.optim.Adam(m.parameters(), 0.1, weight_decay=0.02)
    for _ in range(5):
        for _ in range(20):
            _ = functional.cross_entropy(m(x), y)
            _.backward()
            o.step()
            o.zero_grad()
        if log:
            print(_)
    return m


def uniq(m):
    with torch.no_grad():
        sign = m[0].bias.sign()
        sort = m[0].bias.mul_(sign).argsort()
        m[0].weight = Parameter(m[0].weight.mul_(sign[:, None])[sort])
        m[0].bias = Parameter(m[0].bias[sort])
        m[2].weight = Parameter(m[2].weight.mul_(sign)[:, sort])
        return m


def param(m):
    with torch.no_grad():
        return (
            torch.cat([i.ravel() for i in m.parameters()]).to("cpu", torch.half).numpy()
        )


def f(i):
    x, y = make_classification(128, 4, random_state=i)
    X = torch.tensor(x, dtype=torch.float)
    Y = torch.tensor(y, dtype=torch.long)
    return (
        x.astype("e"),
        y.astype("B"),
        [param(uniq(train(X[:i], Y[:i]))) for i in range(1, len(x) + 1)],
    )

In [ ]:
from pickle import dump

for i in range(100, 200):
    dump(f(i), open(f"nn{i}.pk", "wb"))

In [ ]:
m = Sequential(Linear(34, 64), Tanh(), Linear(64, 30))
c = SmoothL1Loss()
o = torch.optim.Adam(m.parameters(), .01)
x = torch.tensor(np.memmap('nnTransferX.35e', '35e'), dtype=torch.float)
y = torch.tensor(np.memmap('nnTransferY.30e', '30e'), dtype=torch.float)
# for _ in range(100):
#     for _ in range(10):
        